In [1]:
!pip show spicelib
# !pip install spicelib # install if needed

Name: spicelib
Version: 1.4.6
Summary: A set of tools to Automate Spice simulations
Home-page: 
Author: Nuno Brum
Author-email: nuno.brum@gmail.com
License: GPL-3.0
Location: /foss/designs/eda/.venv/lib/python3.12/site-packages
Requires: matplotlib, numpy, psutil, scipy
Required-by: 


In [9]:
from spicelib import SpiceEditor, SimRunner, RawRead
from spicelib.simulators import ngspice_simulator 

import logging
import os

from pathlib import Path


In [3]:
import logging

# Create a logger
logger = logging.getLogger("notebook_logger")
logger.setLevel(logging.DEBUG)  # Set lowest level you want to capture (DEBUG, INFO, WARNING...)

# Create a console handler (for notebook output)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)  # You can choose INFO if DEBUG is too noisy
logger.propagate = False  # stop passing logs to root logger

# Create a formatter
formatter = logging.Formatter(
    fmt="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
console_handler.setFormatter(formatter)

# Add the handler to the logger (avoid duplicates)
if not logger.handlers:
    logger.addHandler(console_handler)

# Example usage
logger.info("Logger initialized.")

22:11:36 [INFO] Logger initialized.


In [ ]:
PATH_TO_NGSPICE = Path("/foss/tools/bin/ngspice")

PROJECT_NAME    = "tia_bpf_1"
SCHEMATIC_NAME  = "tb_ac"

OUTPUT_DIR      = Path(f"./runs/{PROJECT_NAME}/output")
INITIAL_NETLIST = Path(f"../{PROJECT_NAME}/netlist/{SCHEMATIC_NAME}.spice")

os.makedirs(OUTPUT_DIR, exist_ok=True)
if not INITIAL_NETLIST.exists():
    raise FileNotFoundError(f"Initial netlist not found: {INITIAL_NETLIST}")

logger.info(f"Using ngspice from {PATH_TO_NGSPICE}")
logger.info(f"project: {PROJECT_NAME}, schematic: {SCHEMATIC_NAME}")

FileNotFoundError: Initial netlist not found: ../tia_bpf_1/simulation/tb_ac.spice

In [ ]:
runner = SimRunner(
    simulator=ngspice_simulator.NGspiceSimulator.create_from(path_to_exe=PATH_TO_NGSPICE), 
    # simulator=ngspice_simulator.NGspiceSimulator, 
    output_folder=OUTPUT_DIR,
    cwd=OUTPUT_DIR
    )

runner.cwd = "./"

editor = SpiceEditor(netlist_file=INITIAL_NETLIST)

In [ ]:
nodes = editor.get_all_nodes()
nodes

In [ ]:
params = editor.get_all_parameter_names()
params

In [ ]:
# editor.prepare_for_simulator(ngspice_simulator.Simulator)


In [ ]:
editor.set_parameters(VIN=2)
editor.save_netlist("data/saved.spice")

In [ ]:
from typing import List
global_raw_files : List[RawRead] = []
def processing_data(raw_filename, log_filename):
    '''This is a call back function that just prints the filenames'''
    print("Simulation Raw file is %s. The log is %s" % (raw_filename, log_filename))
    # Other code below either using ltsteps.py or raw_read.py
    # log_info = LTSpiceLogReader(log_filename)
    # log_info.read_measures()
    # rise, measures = log_info.dataset["rise_time"]
    global_raw_files.append(RawRead(raw_filename=raw_filename))
    return global_raw_files[raw_filename]

In [ ]:
runtask = runner.run(
    netlist="./data/cs_r_load_ac.spice",
    callback=processing_data,
    # run_filename="runfile.spice",
    exe_log=True )
runtask

In [ ]:
runtask.get_results()

In [ ]:
runner.sim_info()

In [ ]:
raw = global_raw_files[0]
raw.get_trace_names()

In [ ]:
raw.get_plot_names()

In [ ]:
raw.get_plot_name()

In [ ]:
for plot in raw.plots:
    print(plot.get_plot_name())
    print(len(plot.get_trace("v(vout)").get_wave()))
    

In [ ]:
raw.get_trace("v(vout)").get_wave()